## Classic Environment Preflight

This notebook requires the classic runtime. If this check fails, rebuild with `CLASSIC=1 make notebooks-build`, restart the container, and select kernel **Python 3 (classic-langchain)**.


In [ ]:
import os
import sys

def _classic_fail(reason: str) -> None:
    raise RuntimeError(
        f"Classic runtime preflight failed: {reason}\n"
        "Fix:\n"
        "1. CLASSIC=1 make notebooks-build\n"
        "2. make notebooks-up\n"
        "3. In Jupyter, select kernel: Python 3 (classic-langchain)"
    )

kernel_name = os.getenv("JPY_KERNEL_NAME", "")
prefix = sys.prefix.lower()
if "venv-classic" not in prefix and "classic" not in kernel_name.lower():
    _classic_fail(f"detected sys.prefix={sys.prefix!r}, JPY_KERNEL_NAME={kernel_name!r}")

try:
    import langchain  # noqa: F401
except Exception as exc:
    _classic_fail(f"langchain import failed: {exc}")

print("Classic preflight passed.")


Instead of calculating these embeddings everytime , we store these vectors into a vector database. In many cases , they also provide the service to calculate the embeddings using the model you select.

In [ ]:
%pip install -q langchain python-dotenv langchain-openai langchain-community tiktoken chromadb 

In [ ]:
%pip -q install python-dotenv
from dotenv import load_dotenv
load_dotenv()

We first split the markdown doc again

In [ ]:
# We load the texts
from langchain.text_splitter import MarkdownHeaderTextSplitter

history_raw_text = ""
# This is a long document we can split up.
with open("data/history.md") as f:
    history_raw_text = f.read()
    
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
raw_documents = md_splitter.split_text(history_raw_text)

from pprint import pprint
pprint(raw_documents)

And now we use Chromadb as vector database. 
Note: We first reset it as we are running this for demos

In [ ]:

# Resetting chromadb just in case
# For Chroma >=0.6, list_collections returns collection names (strings).

import chromadb

collection_name = "my_langchain"
chroma_client = chromadb.PersistentClient(path="./chromadb")
collection_names = {str(name) for name in chroma_client.list_collections()}
if collection_name in collection_names:
    print("deleting " + collection_name)
    chroma_client.delete_collection(name=collection_name)


Given the embeddings function and given our documents we ask the vector database to take care of this for us.

In [ ]:
# Set the embeddings function
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings()

# Vectory database will calculate them using the embeddings_model provided
# and store the embeddings for each doc in it's database
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=raw_documents,
    embedding=embeddings_model,
    client=chroma_client,
    collection_name=collection_name
    # client_settings
)

Once stored , we can ask it to find the related documents through embeddings.

In [ ]:
query = "Who wrote the Devops Handbook? return the results as json and use the field firstname and lastname"
#Return the result as json and use the field firstname and lastname"
docs = vectorstore.similarity_search_with_relevance_scores(query, k=4, score_threshold=0.7)
pprint(docs)